# Aircraft Intelligence Dashboard — 학습 가이드

> **대상**: Google Cloud 기반 AI 웹 애플리케이션을 처음 접하는 개발자  
> **소요 시간**: 약 2~3시간  
> **난이도**: ⭐⭐⭐ (Python 기초 + GCP 기본 개념 권장)

---

## 이 노트북을 마치면 할 수 있는 것

| # | 학습 목표 | 관련 기술 |
|---|-----------|----------|
| 1 | 항공기 정비 데이터가 어떻게 구조화되어 있는지 이해한다 | BigQuery, SQL |
| 2 | BigQuery에서 쉼표 구분 다중값 컬럼을 올바르게 검색·집계한다 | `UNNEST`, `SPLIT` |
| 3 | FastAPI로 REST API 서버를 구성하는 방법을 설명한다 | Python, FastAPI |
| 4 | Google ADK로 Gemini 기반 AI 에이전트를 만들고 BigQuery와 연결한다 | Google ADK, Gemini |
| 5 | AI 에이전트가 자연어를 SQL로 변환하는 원리를 이해한다 | System Prompt, Tool Use |
| 6 | Docker로 애플리케이션을 컨테이너화하고 Cloud Run에 배포한다 | Docker, Cloud Run |
| 7 | Terraform으로 GCP 인프라를 코드로 선언하고 관리한다 | Terraform, IaC |
| 8 | IAP + Load Balancer로 인증된 사용자만 접근하는 보안 구조를 구성한다 | IAP, HTTPS LB |
| 9 | IAP 토큰을 발급해 배포된 API를 직접 호출한다 | gcloud, REST API |

---

## 전체 학습 흐름

```
① 앱 개요 파악          → 무엇을 왜 만들었는지 큰 그림 이해
        ↓
② 아키텍처 이해          → 컴포넌트 간 데이터 흐름 파악
        ↓
③ 인프라 코드 읽기        → Terraform으로 GCP 리소스 선언하는 방법
        ↓
④ 데이터 다루기           → BigQuery SQL 직접 실행 (실습)
        ↓
⑤ 백엔드 코드 이해        → FastAPI 엔드포인트 구조 분석
        ↓
⑥ AI 에이전트 이해        → ADK + Gemini 에이전트 직접 실행 (실습)
        ↓
⑦ 컨테이너 & 배포 이해    → Docker → Artifact Registry → Cloud Run
        ↓
⑧ 보안 구조 이해          → IAP + Load Balancer 흐름
        ↓
⑨ API 직접 호출           → 실제 배포 서버에 요청 보내기 (실습)
        ↓
⑩ 전체 배포 흐름 정리     → 코드 변경부터 운영까지 한눈에
```

---

## 사전 준비

```bash
# 1. Python 패키지 설치
pip install google-cloud-bigquery google-adk fastapi uvicorn python-dotenv

# 2. GCP 인증
gcloud auth application-default login
gcloud config set project cloud-cycle-pj

# 3. 환경변수 설정 (.env 파일 참고)
export GOOGLE_CLOUD_PROJECT=cloud-cycle-pj
export BIGQUERY_DATASET=mdas_dataset
export BIGQUERY_TABLE=aircraft_dummy
export BIGQUERY_REGION=asia-northeast3
export GOOGLE_CLOUD_LOCATION=us-central1
export GOOGLE_GENAI_USE_VERTEXAI=false
export GOOGLE_API_KEY=<your-api-key>
```

> **실습 셀 구분**: 🟦 파란 배경 셀은 직접 실행 가능한 코드입니다.  
> 환경변수와 GCP 인증이 완료된 상태에서 순서대로 실행하세요.

# Aircraft Intelligence Dashboard 

이 노트북은 항공기 비정형 정비(NR) 데이터를 AI로 분석하는 **Aircraft Intelligence Dashboard**를 처음부터 끝까지 설명합니다.

---

## 목차
1. [애플리케이션 개요](#1-애플리케이션-개요)
2. [전체 아키텍처](#2-전체-아키텍처)
3. [GCP 인프라 (Terraform)](#3-gcp-인프라-terraform)
4. [데이터 레이어 — BigQuery](#4-데이터-레이어--bigquery)
5. [백엔드 — FastAPI](#5-백엔드--fastapi)
6. [AI 에이전트 — Google ADK + Gemini](#6-ai-에이전트--google-adk--gemini)
7. [컨테이너화 — Docker](#7-컨테이너화--docker)
8. [보안 — IAP + Load Balancer](#8-보안--iap--load-balancer)
9. [API 직접 호출해보기](#9-api-직접-호출해보기)
10. [배포 흐름 정리](#10-배포-흐름-정리)

---
## 1. 애플리케이션 개요

### 무엇을 만들었나?

항공기는 운항 중 수많은 **비정형 정비(NR, Non-Routine)** 작업이 발생합니다.  
예: 엔진 오일 누유, APU 이상, 착륙장치 센서 오류 등

이 애플리케이션은:
- BigQuery에 저장된 수만 건의 NR 정비 기록을
- **자연어 채팅**으로 분석하고
- **대시보드/차트**로 시각화하는 웹 서비스입니다.

### 핵심 기능

| 기능 | 설명 |
|------|------|
| AI 채팅 | "APU 결함이 가장 많은 항공사는?" → 자동 SQL 생성 + 결과 분석 |
| 대시보드 | 항공기 기종별 / ATA코드별 / 운항사별 NR 분포 차트 |
| 데이터 테이블 | 원시 정비 기록 페이지네이션 조회 |
| 전문 검색 | 결함명, 조치내용, 컴포넌트 키워드로 전문 검색 |

### 기술 스택 요약

```
Frontend  : Vanilla HTML/CSS/JS + Chart.js v4
Backend   : Python FastAPI + Uvicorn
AI Agent  : Google ADK (Gemini 2.5 Flash)
Database  : Google BigQuery
Container : Docker → Artifact Registry → Cloud Run
Infra     : Terraform (GCP)
Security  : IAP (Identity-Aware Proxy) + HTTPS Load Balancer
```

---
## 2. 전체 아키텍처

```
사용자 브라우저
     │  HTTPS
     ▼
┌──────────────────────────────────────────────────┐
│  Google Cloud Load Balancer  (34.8.37.29)        │
│  └─ SSL 인증서 (nip.io 자동 도메인)               │
│  └─ IAP (Identity-Aware Proxy)                   │  ← 인증된 사용자만 통과
└──────────────────────┬───────────────────────────┘
                       │ (Serverless NEG)
                       ▼
┌──────────────────────────────────────────────────┐
│  Cloud Run  (aircraft-dashboard)                 │
│  Docker 컨테이너: python:3.11-slim               │
│                                                  │
│  FastAPI (app.py)                                │
│  ├── GET  /api/data/summary   ─┐                 │
│  ├── GET  /api/data/charts    ─┼─► BigQuery      │
│  ├── GET  /api/data/table     ─┤   (직접 SQL)    │
│  ├── GET  /api/data/search    ─┘                 │
│  └── POST /api/chat           ──► ADK Runner     │
│                                       │          │
│  Static Files (index.html)    ADK Agent          │
│                                (Gemini 2.5)      │
│                                   │    │         │
│                            BQ Toolset  DA Toolset│
└───────────────────────────────────┼────┼─────────┘
                                    │    │
                      ┌─────────────┘    └──────────────┐
                      ▼                                 ▼
              Google BigQuery                  Conversational
              (mdas_dataset.aircraft_dummy)    Analytics API
```

### 데이터 흐름 — AI 채팅

```
사용자: "APU 결함이 가장 많은 기종은?"
  │
  ▼ POST /api/chat
FastAPI → ADK Runner → Gemini 2.5 Flash
                              │
                              ├─ BigQueryToolset: SQL 자동 생성 & 실행
                              │   SELECT AC_TYPE, COUNT(*) ...
                              │   WHERE COMPONENT_KEYWORD LIKE '%APU%'
                              │
                              ▼
                         응답 텍스트
                         + CHART_DATA:{...}         ← 차트 데이터 포함
                         + SUGGESTED_QUESTIONS:[...]← 후속 질문 제안
  │
  ▼
FastAPI: 응답 파싱 → JSON 반환
브라우저: 텍스트 + Chart.js 인라인 차트 렌더링
```

---
## 3. GCP 인프라 (Terraform)

인프라는 **Terraform**으로 선언적으로 관리합니다. `terraform/` 폴더의 구조:

```
terraform/
├── main.tf              # Provider 설정 (google ~> 5.0)
├── variables.tf         # 변수 선언
├── terraform.tfvars     # 실제 값 주입
├── apis.tf              # GCP API 활성화
├── artifact_registry.tf # Docker 이미지 저장소
├── bigquery.tf          # BQ 데이터셋 + IAM
├── cloud_run.tf         # Cloud Run 서비스
├── iam.tf               # 서비스 계정 + 권한
├── iap.tf               # Load Balancer + IAP + SSL
├── secrets.tf           # Secret Manager (API Key)
└── outputs.tf           # 출력값 (IP, URL 등)
```

### 3-1. 활성화되는 GCP API (`apis.tf`)

```hcl
resource "google_project_service" "apis" {
  for_each = toset([
    "run.googleapis.com",            # Cloud Run
    "artifactregistry.googleapis.com",# Docker 저장소
    "bigquery.googleapis.com",        # 데이터 웨어하우스
    "secretmanager.googleapis.com",  # API 키 보안 저장
    "iap.googleapis.com",            # 인증 프록시
    "compute.googleapis.com",        # Load Balancer
    "iam.googleapis.com",            # 권한 관리
    "cloudbuild.googleapis.com",     # CI/CD
    "iamcredentials.googleapis.com", # Workload Identity
  ])
}
```

### 3-2. 핵심 변수 (`terraform.tfvars`)

```hcl
project_id      = "cloud-cycle-pj"
region          = "us-central1"        # Cloud Run 리전
bigquery_region = "asia-northeast3"    # BQ 리전 (서울)

cloud_run_min_instances = 0            # 비용 절감: 요청 없으면 0으로
cloud_run_max_instances = 3
cloud_run_cpu           = "2"
cloud_run_memory        = "2Gi"

allow_public_access = false            # 조직 정책으로 공개 접근 차단
iap_allowed_members = ["domain:seoeun.altostrat.com"]  # IAP 허용 대상
```

---
## 4. 데이터 레이어 — BigQuery

### 테이블 스키마: `cloud-cycle-pj.mdas_dataset.aircraft_dummy`

| 컬럼 | 타입 | 설명 |
|------|------|------|
| `ID` | INTEGER | 레코드 식별자 |
| `NR_NUMBER` | STRING | 비정형 작업 지시 번호 |
| `MALFUNCTION` | STRING | 결함 설명 (자연어) |
| `CORRECTIVE_ACTION` | STRING | 수행한 조치 내용 |
| `NR_REQUEST_DATE` | DATE | NR 발생일 |
| `AC_TYPE` | STRING | 항공기 기종 (B737, A320 등) |
| `AC_NO` | STRING | 항공기 등록번호 |
| `MSG_NO` | STRING | 메시지 번호 |
| `AMP` | STRING | 운항사 / 정비 프로그램 |
| `COMPONENT_KEYWORD` | STRING | 관련 컴포넌트 키워드 **(쉼표 구분)** |
| `ATA_CODE` | STRING | ATA 챕터 코드 (항공기 시스템 식별) |
| `NR_WORKORDER_NAME` | STRING | 작업 지시 명칭 |

### ⚠️ COMPONENT_KEYWORD 주의사항

`COMPONENT_KEYWORD`는 단일 셀에 **여러 값을 쉼표로** 저장합니다.

```
예시: "ENGINE,APU,FUEL PUMP"
```

따라서 집계/검색 시 반드시 `UNNEST(SPLIT(...))` 로 분리해야 합니다.

In [ ]:
# BigQuery 직접 조회 예시
# 실행 전: pip install google-cloud-bigquery

from google.cloud import bigquery

PROJECT_ID = "cloud-cycle-pj"
DATASET_ID = "mdas_dataset"
TABLE_ID   = "aircraft_dummy"
FULL_TABLE = f"`{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`"

client = bigquery.Client(project=PROJECT_ID, location="asia-northeast3")

# 1) KPI 집계
sql_summary = f"""
SELECT
    COUNT(*)                AS total_records,
    COUNT(DISTINCT AC_NO)   AS total_aircraft,
    COUNT(DISTINCT AC_TYPE) AS aircraft_types,
    COUNT(DISTINCT AMP)     AS operators,
    COUNT(DISTINCT ATA_CODE)AS ata_codes,
    MIN(NR_REQUEST_DATE)    AS earliest_date,
    MAX(NR_REQUEST_DATE)    AS latest_date
FROM {FULL_TABLE}
"""

rows = client.query(sql_summary).result()
for row in rows:
    print(dict(row))

In [ ]:
# 2) COMPONENT_KEYWORD 올바른 집계 방법
# 잘못된 방법: WHERE COMPONENT_KEYWORD LIKE '%APU%'  ← 다른 키워드와 섞여 부정확
# 올바른 방법: UNNEST(SPLIT(..., ','))  ← 각 토큰을 개별로 처리

sql_keyword = f"""
SELECT
    UPPER(TRIM(kw)) AS component,
    COUNT(*)        AS nr_count
FROM {FULL_TABLE},
     UNNEST(SPLIT(COMPONENT_KEYWORD, ',')) AS kw
WHERE TRIM(kw) != ''
GROUP BY component
ORDER BY nr_count DESC
LIMIT 10
"""

rows = client.query(sql_keyword).result()
for row in rows:
    print(f"{row['component']:20s} : {row['nr_count']}건")

In [ ]:
# 3) 전문 검색 쿼리 패턴 (app.py /api/data/search 와 동일)
keyword = "APU"

sql_search = f"""
SELECT *
FROM {FULL_TABLE}
WHERE
  SEARCH(MALFUNCTION, '{keyword}')
  OR SEARCH(CORRECTIVE_ACTION, '{keyword}')
  OR SEARCH(NR_WORKORDER_NAME, '{keyword}')
  OR LOWER(AC_TYPE)   LIKE LOWER('%{keyword}%')
  OR LOWER(AMP)       LIKE LOWER('%{keyword}%')
  OR LOWER(ATA_CODE)  LIKE LOWER('%{keyword}%')
  OR EXISTS (
      SELECT 1
      FROM UNNEST(SPLIT(COMPONENT_KEYWORD, ',')) AS kw
      WHERE TRIM(LOWER(kw)) LIKE LOWER('%{keyword}%')
  )
LIMIT 5
"""

rows = client.query(sql_search).result()
for row in rows:
    print(f"[{row['AC_TYPE']}] {row['MALFUNCTION'][:60]}...")

---
## 5. 백엔드 — FastAPI

리팩토링 후 백엔드는 **레이어드 아키텍처**로 분리되어 있습니다.

### 5-1. 파일 구조

```
app.py          ← FastAPI 앱 bootstrap (lifespan, 라우터 등록)
config.py       ← Settings dataclass (환경변수 → frozen dataclass)
api/
  chat.py       ← POST /api/chat (ADK Runner, 세션 관리, 마커 파싱)
  data.py       ← GET /api/data/* (summary, charts, table, search)
db/
  base.py       ← DataStore ABC (인터페이스 정의)
  bigquery.py   ← BigQueryDataStore 구현 + Gemini 키워드 추출
```

```python
# app.py — 얇은 bootstrap
from config import settings
from db import create_datastore          # 팩토리: DB_TYPE에 따라 구현체 선택
from api.chat import get_runner, router as chat_router
from api.data import router as data_router

@asynccontextmanager
async def lifespan(app):
    app.state.datastore = create_datastore(settings)  # BQ 클라이언트 초기화
    get_runner()   # ADK Runner 워밍업
    yield
```

### 5-2. API 엔드포인트

| 메서드 | 경로 | 설명 |
|--------|------|------|
| `GET` | `/` | SPA `index.html` 반환 |
| `POST` | `/api/chat` | ADK 에이전트와 대화 |
| `GET` | `/api/data/summary` | KPI (총 NR 수, 항공기 수 등) |
| `GET` | `/api/data/charts` | Top-10 분포 차트 데이터 |
| `GET` | `/api/data/table` | 페이지네이션 원시 데이터 |
| `GET` | `/api/data/search` | 전문 검색 (`?q=APU`) |

### 5-3. 채팅 응답 파싱 핵심

에이전트는 응답 텍스트 말미에 특수 마커를 삽입합니다. `_peel_markers()` (`api/chat.py`)가 순서대로 파싱합니다.

```
<분석 텍스트 (마크다운)>
CHART_DATA:{"type":"bar","title":"...","labels":[...],"values":[...]}
SEARCH_DATA:{"keyword":"<검색 키워드>"}
SUGGESTED_QUESTIONS:["질문1","질문2","질문3"]
```

| 마커 | 조건 | 후속 처리 |
|------|------|----------|
| `CHART_DATA` | 통계 데이터가 있을 때 | 프론트엔드 Chart.js 인라인 렌더링 |
| `SEARCH_DATA` | 개별 레코드 키워드 검색 시 | 백엔드가 `BigQueryDataStore.search()` 재실행 → `search_rows` 반환 |
| `SUGGESTED_QUESTIONS` | 항상 | 프론트엔드 추천 질문 버튼 표시 |

FastAPI는 이를 파싱해 구조화된 JSON으로 반환합니다:

```json
{
  "session_id": "uuid",
  "response": "분석 텍스트",
  "chart_data": {"type": "bar", "labels": [...], "values": [...]},
  "suggested_questions": ["질문1", "질문2", "질문3"],
  "search_rows": [{"...":"..."}],
  "search_query": "검색에 사용된 키워드"
}
```

> `search_rows` / `search_query` 는 `SEARCH_DATA` 마커가 있을 때만 포함됩니다.


In [ ]:
# FastAPI 앱 로컬 실행 방법
# 터미널에서:
#   pip install -r requirements.txt
#   uvicorn app:app --host 0.0.0.0 --port 8080 --reload

# 환경변수 (.env)
import os

required_env = {
    "GOOGLE_CLOUD_PROJECT": "cloud-cycle-pj",
    "BIGQUERY_DATASET": "mdas_dataset",
    "BIGQUERY_TABLE": "aircraft_dummy",
    "BIGQUERY_REGION": "asia-northeast3",
    "GOOGLE_CLOUD_LOCATION": "us-central1",
    "GOOGLE_GENAI_USE_VERTEXAI": "false",  # false = Google AI Studio API 사용
    "DB_TYPE": "bigquery",                 # create_datastore() 팩토리 선택자
    "GOOGLE_API_KEY": "<your-key>",        # Cloud Run에서는 Secret Manager로 주입
}

for key, example in required_env.items():
    val = os.getenv(key, "(미설정)")
    status = "✅" if os.getenv(key) else "❌"
    print(f"{status} {key} = {val}")


---
## 6. AI 에이전트 — Google ADK + Gemini

### 6-1. Google ADK란?

**Agent Development Kit (ADK)**은 Google이 제공하는 AI 에이전트 프레임워크입니다.  
LLM(Gemini)에 도구(Tool)를 연결하여 **자율적으로 SQL을 작성하고 실행**할 수 있게 합니다.

```python
from google.adk.agents import Agent
from google.adk.tools.bigquery import BigQueryToolset

root_agent = Agent(
    model="gemini-2.5-flash",
    name="aircraft_agent",
    instruction=SYSTEM_PROMPT,   # 테이블 스키마, 응답 형식 등
    tools=[bq_toolset, da_toolset],
)
```

### 6-2. 에이전트 도구

| 도구 | 역할 |
|------|------|
| `BigQueryToolset` | 자연어 → SQL 자동 생성 및 BigQuery 실행 (최대 200행 반환) |
| `DataAgentToolset` | Conversational Analytics API (CAA) 연동 — 더 복잡한 분석 |

### 6-3. System Prompt 핵심 역할

```
SYSTEM_PROMPT가 에이전트에게 알려주는 것:

① 테이블 스키마 (컬럼명 + 설명)
② COMPONENT_KEYWORD 검색 시 UNNEST/SPLIT 사용 규칙
③ 응답 형식:
   - CHART_DATA:{...} 마커로 차트 데이터 첨부
   - SUGGESTED_QUESTIONS:[...] 마커로 후속 질문 제안
④ 특정 장비 분석 시 포함해야 할 항목
   (NR 건수, 기종별, 운항사별, 월별 트렌드 등)
```

### 6-4. 세션 관리

```python
# 대화 기록은 InMemorySessionService가 session_id로 관리
# session_id가 없으면 uuid4로 새로 생성
# 같은 session_id로 계속 대화하면 이전 문맥 유지

_session_service = InMemorySessionService()

# POST /api/chat 요청마다:
# 1. session_id 확인 또는 신규 생성
# 2. runner.run_async()로 에이전트 실행
# 3. final_response 이벤트에서 응답 텍스트 추출
```

> ⚠️ `InMemorySessionService`는 서버 재시작 시 대화 기록이 사라집니다.  
> 영구 보관이 필요하면 Firestore 기반 세션 서비스로 교체해야 합니다.

In [ ]:
# ADK 에이전트 직접 실행 예시 (adk_runner.py 방식)
# 실행 전: pip install google-adk

import asyncio
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types as genai_types

async def ask_agent(question: str):
    # agent/agent.py 의 root_agent import
    import sys
    sys.path.insert(0, "/home/admin_seoeun_altostrat_com/aircraft")
    from agent import root_agent

    session_service = InMemorySessionService()
    runner = Runner(
        agent=root_agent,
        app_name="notebook_test",
        session_service=session_service,
    )

    session_id = "test-session-001"
    await session_service.create_session(
        app_name="notebook_test", user_id="user", session_id=session_id
    )

    message = genai_types.Content(
        role="user",
        parts=[genai_types.Part(text=question)],
    )

    response_text = ""
    async for event in runner.run_async(
        user_id="user", session_id=session_id, new_message=message
    ):
        if event.is_final_response() and event.content:
            response_text = "".join(
                p.text for p in event.content.parts if hasattr(p, "text") and p.text
            )

    return response_text


# 실행 (주석 해제 후 사용)
# result = await ask_agent("전체 NR 건수와 항공기 기종별 분포를 알려줘")
# print(result[:500])

---
## 7. 컨테이너화 — Docker

### Dockerfile 해설

```dockerfile
FROM python:3.11-slim          # 경량 베이스 이미지

WORKDIR /app

# 시스템 패키지 (curl: 헬스체크용)
RUN apt-get update && apt-get install -y --no-install-recommends curl

# 의존성 먼저 설치 (레이어 캐시 활용)
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# 소스코드 복사 (레이어드 아키텍처 반영)
COPY app.py .
COPY config.py .        # Settings dataclass
COPY agent/ agent/      # ADK Agent + System Prompt
COPY api/ api/          # chat.py, data.py 라우터
COPY db/ db/            # DataStore 인터페이스 + BigQuery 구현체
COPY static/ static/    # index.html, chart.min.js

ENV PYTHONUNBUFFERED=1
ENV PORT=8080
EXPOSE 8080

# ASGI 서버로 실행 (workers=1: Cloud Run은 컨테이너당 1프로세스 권장)
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8080", "--workers", "1"]
```

### 빌드 → 푸시 → 배포 명령어

```bash
# 1) Docker 인증
gcloud auth configure-docker us-central1-docker.pkg.dev

# 2) 이미지 빌드 & 푸시
docker build -t us-central1-docker.pkg.dev/cloud-cycle-pj/aircraft-dashboard-images/aircraft-dashboard:latest .
docker push us-central1-docker.pkg.dev/cloud-cycle-pj/aircraft-dashboard-images/aircraft-dashboard:latest

# 3) Cloud Run 배포
gcloud run deploy aircraft-dashboard \
  --image=us-central1-docker.pkg.dev/cloud-cycle-pj/aircraft-dashboard-images/aircraft-dashboard:latest \
  --region=us-central1
```

> Cloud Run은 이미지 버전을 CI/CD(Cloud Build)가 관리하므로  
> Terraform의 `cloud_run.tf`에는 `lifecycle { ignore_changes = [image] }` 가 설정되어 있습니다.


---
## 8. 보안 — IAP + Load Balancer

### 왜 IAP를 사용하나?

조직 정책 `constraints/iam.allowedPolicyMemberDomains`에 의해  
`allUsers` (공개 접근)가 차단되어 있습니다.  
따라서 **IAP(Identity-Aware Proxy)**를 통해 인증된 사용자만 접근 허용합니다.

### 구성 요소

```
인터넷
  │ :80 (HTTP)
  ▼
Forwarding Rule (lb_http)
  └─ Target HTTP Proxy
       └─ URL Map (http_redirect)
            └─ 301 redirect → HTTPS

  │ :443 (HTTPS)
  ▼
Forwarding Rule (lb_https)  ← IP: 34.8.37.29
  └─ Target HTTPS Proxy
       └─ SSL Certificate (Google 관리형, nip.io 도메인)
            └─ URL Map → Backend Service
                              │
                        IAP 활성화
                        (OAuth 클라이언트 인증)
                              │
                        Serverless NEG
                              │
                        Cloud Run (내부 트래픽만 허용)
```

### IAP 접근 제어 (`iap.tf`)

```hcl
resource "google_iap_web_backend_service_iam_binding" "iap_access" {
  role    = "roles/iap.httpsResourceAccessor"
  members = ["domain:seoeun.altostrat.com"]  # 도메인 전체 허용
}
```

### nip.io 자동 도메인

커스텀 도메인 없이도 IP 기반 HTTPS를 사용할 수 있습니다:

```
IP: 34.8.37.29
→ 도메인: 34-8-37-29.nip.io  (점을 하이픈으로 치환)
→ URL: https://34-8-37-29.nip.io
```

nip.io는 IP를 그대로 반환하는 DNS 서비스이므로 별도 DNS 설정 불필요합니다.

### API Key 보안 — Secret Manager

```hcl
# secrets.tf: API 키를 Secret Manager에 저장
resource "google_secret_manager_secret" "api_key" { ... }

# cloud_run.tf: 컨테이너에 환경변수로 주입 (평문 노출 없음)
env {
  name = "GOOGLE_API_KEY"
  value_source {
    secret_key_ref {
      secret  = google_secret_manager_secret.api_key.secret_id
      version = "latest"
    }
  }
}
```

---
## 9. API 직접 호출해보기

IAP가 활성화되어 있으므로 API 호출 시 **IAP 토큰**이 필요합니다.

In [ ]:
import subprocess, json, urllib.request

IAP_URL = "https://34-8-37-29.nip.io"
CLOUD_RUN_URL = "https://aircraft-dashboard-6epkbwshrq-uc.a.run.app"

def get_iap_token(target_audience: str) -> str:
    """현재 gcloud 계정의 IAP ID 토큰 발급"""
    result = subprocess.run(
        ["gcloud", "auth", "print-identity-token",
         f"--audiences={target_audience}"],
        capture_output=True, text=True
    )
    return result.stdout.strip()


def call_api(path: str, method: str = "GET", body: dict = None) -> dict:
    """IAP 인증 헤더를 포함한 API 호출"""
    # IAP OAuth 클라이언트 ID (iap.tf의 google_iap_client)
    IAP_CLIENT_ID = "789490500980-54omubmde0qgimjc6osshu8cnd16oe5l.apps.googleusercontent.com"
    token = get_iap_token(IAP_CLIENT_ID)

    url = IAP_URL + path
    data = json.dumps(body).encode() if body else None
    req = urllib.request.Request(
        url, data=data, method=method,
        headers={
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json",
        }
    )
    with urllib.request.urlopen(req) as resp:
        return json.loads(resp.read())


print("API 호출 준비 완료")

In [ ]:
# KPI 요약 조회
summary = call_api("/api/data/summary")
print("=== KPI Summary ===")
for k, v in summary.items():
    print(f"  {k:20s}: {v}")

In [ ]:
# 차트 데이터 조회
charts = call_api("/api/data/charts")

print("=== 항공기 기종 Top 5 ===")
for item in charts["aircraft_type"][:5]:
    bar = "█" * (item["value"] // 10)
    print(f"  {item['label']:10s} {bar} {item['value']}")

print("\n=== 컴포넌트 키워드 Top 5 ===")
for item in charts["component_keyword"][:5]:
    print(f"  {item['label']:20s}: {item['value']}건")

In [ ]:
# AI 채팅 호출
chat_resp = call_api("/api/chat", method="POST", body={
    "message": "ATA 코드별 NR 건수 상위 5개를 알려줘"
})

print("=== AI 응답 ===")
print(chat_resp["response"][:500])

if chat_resp.get("chart_data"):
    print("\n=== 차트 데이터 ===")
    print(json.dumps(chat_resp["chart_data"], ensure_ascii=False, indent=2))

if chat_resp.get("suggested_questions"):
    print("\n=== 추천 질문 ===")
    for q in chat_resp["suggested_questions"]:
        print(f"  • {q}")

if chat_resp.get("search_rows"):
    print(f"\n=== 키워드 검색 결과 ({chat_resp.get('search_query', '')}) ===")
    for row in chat_resp["search_rows"][:3]:  # 상위 3건만 출력
        print(f"  [{row.get('AC_TYPE','')}] {str(row.get('MALFUNCTION',''))[:60]}...")
    print(f"  (총 {len(chat_resp['search_rows'])}건)")


---
## 10. 배포 흐름 정리

```
코드 변경
   │
   ▼
docker build & push
   │  us-central1-docker.pkg.dev/cloud-cycle-pj/aircraft-dashboard-images/aircraft-dashboard:latest
   ▼
gcloud run deploy  (또는 Cloud Build 자동화)
   │
   ▼
Cloud Run 새 리비전 배포 (무중단)
   │
   ▼
Load Balancer → IAP → Cloud Run (새 컨테이너)
```

### 인프라 변경 시

```bash
cd terraform/
terraform plan    # 변경 미리 확인
terraform apply   # 적용
```

### 현재 배포 정보

| 항목 | 값 |
|------|----|
| IAP URL | `https://34-8-37-29.nip.io` |
| Cloud Run URL | `https://aircraft-dashboard-6epkbwshrq-uc.a.run.app` |
| LB IP | `34.8.37.29` |
| Artifact Registry | `us-central1-docker.pkg.dev/cloud-cycle-pj/aircraft-dashboard-images` |
| BigQuery 테이블 | `cloud-cycle-pj.mdas_dataset.aircraft_dummy` |
| 서비스 계정 | `aircraft-dashboard-sa@cloud-cycle-pj.iam.gserviceaccount.com` |

### IAP 접근 허용 도메인

```hcl
iap_allowed_members = ["domain:seoeun.altostrat.com"]
# seoeun.altostrat.com 도메인의 모든 사용자 접근 가능
```

---

## 참고 명령어 모음

```bash
# Cloud Run 로그 확인
gcloud run services logs read aircraft-dashboard --region=us-central1 --limit=50

# Cloud Run 현재 상태
gcloud run services describe aircraft-dashboard --region=us-central1

# BigQuery 빠른 조회
bq query --use_legacy_sql=false \
  'SELECT COUNT(*) FROM `cloud-cycle-pj.mdas_dataset.aircraft_dummy`'

# Terraform 상태 확인
cd terraform && terraform show
```